# 02 · Feature engineering: what can a credit decision legitimately use?

**Owner:** Credit Risk Analyst · **Audience:** Model Validation, Compliance · **Inputs:** raw Home Credit 2024 tables via `sql/01` to `sql/06`

**How this notebook is laid out:** the first code cells build the tables and run every computation; the TL;DR is
generated from those results, and the sections after it show the evidence.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

from src import charts
from src.config import load_config, processed
from src.db import connect, run_sql_file
from src.fairness import proxy_strength
from src.features import CATALOG, CATEGORICAL, FEATURES, GROUPS, LABELS, NUMERIC, catalog_frame, check_against_table
from src.metrics import gini, iv_categorical, iv_numeric, psi
from src.modeling import prepare_lgbm_frame, train_lgbm

pio.renderers.default = "notebook+png"
pd.set_option("display.max_colwidth", 140)
cfg = load_config()
SEED = cfg["seed"]
OUT = processed(cfg, "dashboard")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Build the feature tables (sql/01 to sql/06) and load them

con = connect(cfg)
for f in ["sql/01_stg_base.sql", "sql/02_stg_static.sql", "sql/03_int_applprev_pit.sql", "sql/04_int_person_applicant.sql",
          "sql/05_fct_application_features.sql", "sql/06_abl_snapshot_as_provided.sql"]:
    run_sql_file(con, ROOT / f)
recon = con.execute("""
    SELECT 'stg_base' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT case_id) AS cases FROM stg_base
    UNION ALL SELECT 'stg_static', COUNT(*), COUNT(DISTINCT case_id) FROM stg_static
    UNION ALL SELECT 'int_applprev_pit', COUNT(*), COUNT(DISTINCT case_id) FROM int_applprev_pit
    UNION ALL SELECT 'int_person_applicant', COUNT(*), COUNT(DISTINCT case_id) FROM int_person_applicant
    UNION ALL SELECT 'fct_application_features', COUNT(*), COUNT(DISTINCT case_id) FROM fct_application_features
    UNION ALL SELECT 'abl_features_as_provided', COUNT(*), COUNT(DISTINCT case_id) FROM abl_features_as_provided""").df()
split_recon = con.execute("""SELECT split, COUNT(*) AS loans, SUM(target) AS bads, AVG(target) AS bad_rate
                              FROM fct_application_features GROUP BY split ORDER BY MIN(date_decision)""").df()
cols = con.execute("DESCRIBE fct_application_features").df().column_name.tolist()
check_against_table(cols)  # SQL table and feature catalog must agree exactly
df = con.execute("SELECT * FROM fct_application_features ORDER BY case_id").df()
abl = con.execute("SELECT * FROM abl_features_as_provided ORDER BY case_id").df()
con.close()
assert (recon.rows == 1_526_659).all() and (recon.cases == 1_526_659).all(), "every table must have exactly one row per booked loan"

In [ ]:
# Feature inclusion log summary

log = pd.read_csv(ROOT / "docs" / "feature_inclusion_log.csv")
rule_names = {
    "OK": "usable as provided", "L2": "dated field after decision", "L3": "describes a post-decision event", "L4": "short window or current status (snapshot at decision + 14d)",
    "L5": "arrives after decision (tax authority)", "L1": "identifier", "FAIR": "fair-lending exclusion", "FAIRNESS": "fair-lending exclusion", "DUP": "duplicate of a kept field",
    "SPARSE": "almost empty or constant", "LOWSIG": "no measurable signal", "OTHER": "unstable coverage or meaning", "PIT_AGG": "used in point-in-time aggregates",
    "PROCESS": "output of the lender's own pricing / targeting", "ROLLOUT": "collected only from mid-2019", "L6": "flagged for review",
}
summary = (log.assign(rule_text=log.rule.map(rule_names).fillna(log.rule))
              .groupby(["decision", "rule_text"]).size().rename("columns").reset_index()
              .sort_values(["decision", "columns"], ascending=[True, False]))

In [ ]:
# Univariate screening: IV on train, missing share, stability train -> test

tr, te = df[df.split == "train"], df[df.split == "test"]
rows = []
for f in FEATURES:
    x_tr, x_te = tr[f], te[f]
    if f in CATEGORICAL:
        iv = iv_categorical(x_tr.astype(object).values, tr.target.values)
        stab = np.nan
    else:
        iv = iv_numeric(x_tr.astype(float).values, tr.target.values, n_bins=10)
        stab = psi(x_tr.astype(float).values, x_te.astype(float).values)
    rows.append({"feature": f, "label": LABELS[f], "group": GROUPS[f], "iv_train": iv, "missing_share_train": x_tr.isna().mean(), "psi_train_to_test": stab})
feat = pd.DataFrame(rows).sort_values("iv_train", ascending=False).reset_index(drop=True)
feat.to_parquet(OUT / "feature_screening.parquet", index=False)

In [ ]:
# Leakage ablation: point-in-time (A) vs snapshot as provided (B)

b_cols = [c for c in abl.columns if c != "case_id"]
abl_feats = list(dict.fromkeys([f for f in FEATURES if f not in b_cols] + b_cols))
abl_cat = CATEGORICAL + ["raw_last_status", "raw_last_reject_reason", "raw_last_cancel_reason"]
dfB = df[["case_id", "split", "target"] + [f for f in FEATURES if f not in b_cols]].merge(abl, on="case_id", how="left")
assert (dfB.case_id.values == df.case_id.values).all()

def fit_eval(frame, feats, cats, label):
    trn, val = frame[frame.split == "train"], frame[frame.split == "valid"]
    Xtr, categories = prepare_lgbm_frame(trn, feats, cats)
    Xva, _ = prepare_lgbm_frame(val, feats, cats, categories)
    booster = train_lgbm(Xtr, trn.target.values, Xva, val.target.values, seed=SEED)
    return booster, categories

boosterA, catsA = fit_eval(df, FEATURES, CATEGORICAL, "A")
boosterB, catsB = fit_eval(dfB, abl_feats, abl_cat, "B")

# decision-time version of B's inputs: substitute every point-in-time rebuild we have
dfB_pit = dfB.copy()
for c in [c for c in b_cols if c in FEATURES]:
    dfB_pit[c] = df[c].values  # ap_* aggregates and capped DPD maxima
subst = {"raw_days_since_last_application": "ap_days_since_last_app", "raw_days_since_last_rejection": "ap_days_since_last_reject",
         "raw_days_since_last_approval": "ap_days_since_last_approval", "raw_applications_30d": "ap_n_apps_30d",
         "raw_last_reject_amount": "ap_last_rej_credamount", "raw_last_approved_amount": "ap_last_appr_credamount"}
for raw, pit_col in subst.items():
    dfB_pit[raw] = df[pit_col].values
for c in ["raw_days_since_last_activation", "raw_days_since_last_delinquency", "raw_days_since_last_unpaid", "raw_days_since_worst_instalment", "raw_days_since_first_due"]:
    dfB_pit.loc[dfB_pit[c] < 0, c] = np.nan  # an event after the decision cannot be seen at decision time

# raw predictions of the three variants on validation and test
from src.economics import Economics
from src.modeling import PlattCalibrator
from src.strategy import approve_ev_positive, evaluate_policy

econ = Economics.from_config(cfg)
pred = {}
for split in ["valid", "test"]:
    m = (df.split == split).to_numpy()
    pred[(split, "A")] = boosterA.predict(prepare_lgbm_frame(df[m], FEATURES, CATEGORICAL, catsA)[0])
    pred[(split, "B")] = boosterB.predict(prepare_lgbm_frame(dfB[m], abl_feats, abl_cat, catsB)[0])
    pred[(split, "Bp")] = boosterB.predict(prepare_lgbm_frame(dfB_pit[m], abl_feats, abl_cat, catsB)[0])
names = {"A": "A · point-in-time (ours)", "B": "B · as provided, offline test", "Bp": "B · as provided, decision-time inputs"}
res = []
for split in ["valid", "test"]:
    yy = df.loc[df.split == split, "target"].to_numpy()
    for k in ["A", "B", "Bp"]:
        res.append({"split": split, "model": names[k], "gini": gini(yy, pred[(split, k)])})
ablation = pd.DataFrame(res)
ablation.to_parquet(OUT / "leakage_ablation.parquet", index=False)

# where does the naive model's edge sit? loans with client activity dated after the decision
mt = (df.split == "test").to_numpy()
yt = df.loc[mt, "target"].to_numpy()
ab_t = abl.loc[mt]
post = ((ab_t.raw_days_since_last_application < 0) | (ab_t.raw_days_since_last_rejection < 0) | (ab_t.raw_days_since_last_approval < 0)
        | (ab_t.raw_days_since_last_activation < 0) | (ab_t.raw_days_since_last_delinquency < 0)).fillna(False).to_numpy(dtype=bool)
sub_rows = []
for grp, gm in [("Activity dated after the decision", post), ("No post-decision activity", ~post)]:
    for k in ["A", "B", "Bp"]:
        sub_rows.append({"group": grp, "share_of_test": gm.mean(), "bad_rate": yt[gm].mean(), "model": names[k], "gini": gini(yt[gm], pred[("test", k)][gm])})
ablation_groups = pd.DataFrame(sub_rows)
ablation_groups.to_parquet(OUT / "leakage_ablation_groups.parquet", index=False)

# money: what the naive model promises offline vs what it delivers with decision-time inputs (same cost-based rule)
mv = (df.split == "valid").to_numpy()
gain_t, loss_t = econ.per_loan(df.loc[mt, "loan_amount"], df.loc[mt, "product_type"])
calA = PlattCalibrator().fit(pred[("valid", "A")], df.loc[mv, "target"])
calB = PlattCalibrator().fit(pred[("valid", "B")], df.loc[mv, "target"])
biz_rows = []
for label, k, cal in [("A · point-in-time (ours)", "A", calA), ("B · what the offline test promises", "B", calB), ("B · what it delivers at decision time", "Bp", calB)]:
    pdv = cal.transform(pred[("test", k)])
    a = approve_ev_positive(pdv, gain_t, loss_t)
    r = evaluate_policy(a, yt, gain_t, loss_t)
    biz_rows.append({"model": label, "approval_rate": r["approval_rate"], "expected_bad_rate_of_approved": pdv[a].mean(),
                     "observed_bad_rate_of_approved": r["bad_rate_approved"], "net_profit": r["net_profit"]})
leak_business = pd.DataFrame(biz_rows)
leak_business.to_parquet(OUT / "leakage_business.parquet", index=False)
gA, gB, gBp = ablation[ablation.split == "test"].gini.tolist()  # test Gini: A, B offline, B at decision time

In [ ]:
# Where does model B's extra signal come from?

imp = pd.DataFrame({"feature": boosterB.feature_name(), "gain": boosterB.feature_importance("gain")})
imp["share_of_gain"] = imp.gain / imp.gain.sum()
leak_like = set(b_cols)  # every column whose naive version differs from ours or that we dropped
imp["naive_or_dropped_input"] = imp.feature.isin(leak_like)
top_b = imp.sort_values("gain", ascending=False).head(15)

In [ ]:
# Fair-lending proxy check

Xall, _ = prepare_lgbm_frame(df[df.split == "train"], FEATURES, CATEGORICAL)
s_train = df.loc[df.split == "train"]
proxy = pd.DataFrame([
    {"attribute": "Sex (female vs male)", "auc_from_model_inputs": proxy_strength(Xall, s_train.fair_sex.values, seed=SEED)},
    {"attribute": "Age 50 or over", "auc_from_model_inputs": proxy_strength(Xall, np.where(s_train.fair_age_years >= 50, "50+", "<50"), seed=SEED)},
])
proxy.to_parquet(OUT / "proxy_check.parquet", index=False)

In [ ]:
# Summary for the TL;DR at the top
n_keep, n_rebuild, n_drop = (log.decision == "keep").sum(), (log.decision == "rebuild_pit").sum(), (log.decision == "drop").sum()
ivs = feat.set_index("feature").iv_train
unstable = feat[feat.psi_train_to_test > 0.10]
ag = ablation_groups.set_index(["group", "model"]).gini
gsub_A, gsub_B = ag[("Activity dated after the decision", names["A"])], ag[("Activity dated after the decision", names["B"])]
summary_json = {
    "raw_columns_reviewed": int(len(log)), "kept": int(n_keep), "rebuilt": int(n_rebuild), "dropped": int(n_drop),
    "n_features": len(FEATURES), "gini_test_pit": gA, "gini_test_naive_offline": gB, "gini_test_naive_decision_time": gBp, "share_post_decision_activity": float(post.mean()), "gini_post_activity_pit": float(gsub_A), "gini_post_activity_naive": float(gsub_B),
    "proxy_auc_sex": float(proxy.auc_from_model_inputs[0]), "proxy_auc_age50": float(proxy.auc_from_model_inputs[1]),
    "n_unstable_features": int(len(unstable)),
}
(processed(cfg, "nb02_summary.json")).write_text(json.dumps(summary_json, indent=2))
display(Markdown(f"""
## TL;DR

- **{summary_json['raw_columns_reviewed']} raw columns reviewed one by one**; {n_keep} kept as provided, {n_rebuild} rebuilt point-in-time from the previous-applications
  table, {n_drop} dropped (each with a written reason in `docs/feature_inclusion_log.csv`). Result: **{len(FEATURES)} features** in six business groups,
  one row per loan, reconciled to the 1,526,659 booked loans.
- **The leak is real but modest overall, and concentrated.** Using the snapshot fields as provided, the same model would report an out-of-time Gini
  of **{gB:.3f}** instead of **{gA:.3f}**. The gap sits in the **{post.mean():.1%}** of test loans whose client did something *after* the decision
  (applied, was rejected, fell behind elsewhere): there the naive model scores **{gsub_B:.3f}** vs **{gsub_A:.3f}**. That information does not exist when the
  decision is made. Part of the naive model's edge comes from fields that cannot be rebuilt as of the decision date (so they were dropped rather than trusted).
- **Fair lending:** protected and proxy attributes are excluded, but the remaining inputs still predict sex with AUC **{proxy.auc_from_model_inputs[0]:.2f}** and
  age 50+ with AUC **{proxy.auc_from_model_inputs[1]:.2f}**, so outcomes are tested by group in notebook 04.
- {len(unstable)} of the {len(FEATURES) - len(CATEGORICAL)} numeric features shift between training and test (PSI > 0.10); they are tracked in the monitoring plan.
"""))

## 1. Question

Which inputs can a credit decision legitimately use at the moment of the decision, and what would it cost to get this
wrong? Two failure modes matter: information from *after* the decision (it flatters the model in testing and fails in
production) and inputs that are, or stand in for, protected characteristics.

## 2. Data and method

Five raw tables (see notebook 01) are turned into one row per loan by six SQL files. Every raw column was reviewed
against written rules; the decisions are logged in `docs/feature_inclusion_log.csv`. Features are screened with
Information Value (signal) and PSI (stability), then the effect of the leak is measured with an ablation.

## 3. Results
### 3.1 Reconciliation

In [ ]:
display(recon)
display(split_recon)

Every table has exactly one row per booked loan and the split totals match `stg_base`, so no joins duplicated or dropped
loans.

### 3.2 Which raw fields made it in, and why

All 294 raw columns from the five tables were reviewed one by one against written rules (point-in-time, stability, fair
lending, duplication, coverage). The full log with a reason per column is `docs/feature_inclusion_log.csv`.

In [ ]:
display(log.groupby("decision").size().rename("raw columns").to_frame())
summary

### 3.3 The model-ready features

90 features in six business groups. Information Value (IV) is measured on the training window only; PSI compares the
training and out-of-time test distributions (below 0.10 = stable).

In [ ]:
display(feat.groupby("group").agg(features=("feature", "size"), best_iv=("iv_train", "max")).sort_values("best_iv", ascending=False))
feat.head(25)

In [ ]:
top = feat.head(20).iloc[::-1]
fig = go.Figure(go.Bar(x=top.iv_train, y=top.label, orientation="h", marker_color=charts.BLUE, width=0.6,
                       customdata=np.c_[top.feature, top.group], hovertemplate="%{y}<br>%{customdata[0]} (%{customdata[1]})<br>IV %{x:.3f}<extra></extra>"))
charts.base_layout(fig, "Repayment behaviour on past loans carries most of the univariate signal",
                   "Information Value on the training window, top 20 of 90 features (0.1 = medium, 0.3 = strong)", height=560, legend=False)
fig.update_layout(margin=dict(l=300))
fig

### 3.4 What the look-ahead leak would have done (ablation)

Same algorithm, same parameters, same time split; only the inputs differ:
- **A · point-in-time (ours):** the 90 features above.
- **B · as provided (naive):** previous-application aggregates computed over *all* rows including applications made after
  the decision, raw (uncapped) days-past-due, raw bureau enquiry windows, plus the dropped snapshot fields (dated "last
  event" fields, short-window and current-status fields).

Offline, B's extra information exists in the test data too, so a naive validation cannot detect the problem. To estimate
what B would really deliver, we also score B's model on **decision-time inputs**: every field we can rebuild point-in-time
is swapped for its point-in-time value. Fields with no rebuild are left as provided, so this estimate is still optimistic
for B.

In [ ]:
ablation.pivot(index="model", columns="split", values="gini").round(4)

In [ ]:
t = ablation[ablation.split == "test"].reset_index(drop=True)
colors = [charts.BLUE, charts.ORANGE, charts.NEUTRAL]
fig = go.Figure(go.Bar(x=t.model, y=t.gini, marker_color=colors, width=0.5, text=[f"{v:.3f}" for v in t.gini], textposition="outside",
                       textfont=dict(color=charts.INK), hovertemplate="%{x}<br>Gini %{y:.4f}<extra></extra>"))
gA, gB, gBp = t.gini.tolist()
charts.base_layout(fig, f"Using the snapshot as provided would overstate the model by {100 * (gB - gA):.1f} Gini points offline",
                   "Out-of-time test Gini (Jan to Mar 2020) of the same LightGBM set-up with different inputs", height=420, legend=False)
fig.update_yaxes(range=[max(0, min(t.gini) - 0.08), max(t.gini) + 0.03], title="Gini")
fig

In [ ]:
display(Markdown(f"**{imp.loc[imp.naive_or_dropped_input, 'share_of_gain'].sum():.0%}** of model B's total gain comes from inputs that are "
                 "either computed differently by us (point-in-time) or dropped. Top 15 inputs of model B:"))
top_b[["feature", "share_of_gain", "naive_or_dropped_input"]]

**Where the naive model's edge comes from.** Split the test window by whether the client did something *after* the
decision date that the naive inputs can see (a new application, a rejection, an approval or a delinquency elsewhere):

In [ ]:
ablation_groups.pivot_table(index="model", columns="group", values="gini").round(4)

**What it would mean for the business.** Same cost-based cut-off rule (approve when expected profit is positive),
each model calibrated on its own validation predictions:

In [ ]:
leak_business

**Reading the result.** The leak flatters the naive model by about one Gini point overall, which on its own would look
like a reason to keep the extra fields. It is not: the gain is concentrated in the minority of loans whose client did
something after the decision, i.e. it is information the lender cannot have when deciding. Swapping in decision-time
values where we can rebuild them removes part of the gain; the rest comes from snapshot fields that cannot be rebuilt
as of the decision date, so their real production value is unknowable from this data. We drop them rather than ship a
model whose test results we cannot reproduce in production. The overall effect is modest here because most of the
signal comes from long repayment histories, but the same leak in a short-history product would be much larger.

### 3.5 Fair-lending exclusions and proxy check

Sex, age, marital and family status, number of children, education, language, income type (it contains pension and
disability categories) and addresses are excluded from the inputs. Exclusion alone does not guarantee fair outcomes:
other inputs can act as proxies. The check asks how well the 90 model inputs can predict a sensitive attribute
(AUC 0.5 = no information).

In [ ]:
proxy

The inputs carry some information about both attributes (mostly through product choice, loan size, income and
repayment history, which differ by life stage). That is expected and not by itself discriminatory, but it means the
policy's outcomes must be tested by group, which notebook 04 does (approval-rate ratio, bad rate and calibration by sex
and age band).

## 4. Limitations

- Long-window repayment summaries (6 to 24 months) are still measured at decision + 14 days; the residual inflation is
  small (at most about 0.01 IV per field) and documented in `docs/03_data_quality_checks.md`.
- Reason codes of previous rejections and cancellations are hashed in the source data; customer-facing decline text would
  need the lender's code list.
- Bureau enquiry counts were not collected for about 45% of January 2019 applications (6% from June 2019); the missing
  group therefore mixes "not collected" and "no record" in early training months.

## 5. Decisions carried into modelling

| Decision | Why |
|---|---|
| Use the 90 point-in-time features for both the champion scorecard and the challenger | the only inputs that exist at decision time |
| Report the naive model's Gini only as a warning, never as a result | its extra Gini is post-decision information |
| Test approval rates, bad rates and calibration by sex and age band | inputs still carry some information about both |
| Track the features that shift between train and test in the monitoring plan | early warning before performance drops |